In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("table_name","")

In [0]:
table_name = dbutils.widgets.get('table_name')


In [0]:
dbutils.widgets.text("catalog_name","")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
print(catalog_name)
print(table_name)

In [0]:
first_schema = dbutils.fs.ls(f"s3://briandbproject/raw_data/{table_name}")
static_df = spark.read.parquet(f"s3://briandbproject/raw_data/{table_name}/{first_schema[0].name}")
inferred_schema = static_df.schema

In [0]:
df = spark.readStream\
    .format("parquet")\
    .schema(inferred_schema)\
    .option("mergeSchema", "true")\
    .load(f"s3://briandbproject/raw_data/{table_name}")\
    .withColumn("created_at",current_timestamp()) # add created time 
            

In [0]:
spark.sql(f'create schema if not exists {catalog_name}.bronze')

In [0]:
# Streaming schema
df_schema = df.schema
empty_df = spark.createDataFrame([],df_schema)
# prepare columns
sql_columns = ',\n'.join([
    f"`{field.name}` {field.dataType.simpleString()}"
    for field in df_schema.fields
])
#create table
if spark.catalog.tableExists(f'{catalog_name}.bronze.{table_name.split("_")[1]}'):
  print("Table already exists")
else:
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS {catalog_name}.bronze.{table_name.split('_')[1]} (
                  `id` BIGINT GENERATED ALWAYS AS IDENTITY,
                  {sql_columns}
              )""")
        

In [0]:
df.writeStream\
  .format("delta")\
  .outputMode("append")\
  .option("checkpointLocation",f"s3://briandbproject/raw_data/checkpoints/{catalog_name}/{table_name.split('_')[1]}")\
  .trigger(once=True)\
  .toTable(f"{catalog_name}.bronze.{table_name.split('_')[1]}")